In [ ]:
from collections import defaultdict
import os
import re

import pandas as pd

In [ ]:
dataset_dir = "path/to/shadow_dataset"
crops_dir = os.path.join(dataset_dir, "crops")
root_dir = os.path.join(dataset_dir, "root")
dsm_dir = os.path.join(dataset_dir, "dsm")

cities = ["JAX", "OMA", "UCSD"]

## DSMs stats

In [ ]:
data = []

for aoi in os.listdir(dsm_dir):
    city_name = aoi.split("_")[0]
    tile_id = aoi.split("_")[1]
    aoi_path = os.path.join(dsm_dir, aoi)
    if not os.path.isdir(aoi_path):
        continue
    for dsm in os.listdir(aoi_path):
        if dsm.endswith(".tif"):
            # Split by underscore and take parts after city and tile_id
            parts = dsm.split("_")
            if len(parts) >= 3:
                # Join all middle parts until '_dsm.tif'
                description = "_".join(parts[2:]).replace("_dsm.tif", "")
                
                # Extract year if present
                year_match = re.search(r'(\d{4})', dsm)
                year = year_match.group(1) if year_match else None
                
                record = {
                    "city": city_name,
                    "tile_id": tile_id,
                    "dsm_name": dsm,
                    "lidar_name": description,
                    "year": year,
                    "full_path": os.path.join(aoi_path, dsm)
                }
                data.append(record)

dsms_df = pd.DataFrame(data)

In [ ]:
# Count how many city-tile pairs we have
tile_counts = dsms_df.groupby(["city", "tile_id"]).size().reset_index(name="count")
print(tile_counts)

In [ ]:
# Count how many dsms for each city
city_counts = dsms_df.groupby("city").size().reset_index(name="count")
print(city_counts)

In [ ]:
# Count how many tiles for each city
city_tile_counts = dsms_df.groupby("city")["tile_id"].nunique().reset_index(name="count")
print(city_tile_counts)

In [ ]:
# See how many different lidar names we have
lidar_counts = dsms_df.groupby("lidar_name").size().reset_index(name="count")
print(lidar_counts)

In [ ]:
# See from which years we have data
year_counts = dsms_df.groupby("year").size().reset_index(name="count")
print(year_counts)


## Crops stats

In [ ]:
data = []

for aoi in os.listdir(crops_dir):
    city_name = aoi.split("_")[0]
    tile_id = aoi.split("_")[1]
    aoi_path = os.path.join(crops_dir, aoi)
    if not os.path.isdir(aoi_path):
        continue
    for crop in os.listdir(aoi_path):
        if crop.endswith(".tif"):
            # Split by underscore and take parts after city and tile_id
            parts = crop.split("_")
            if len(parts) >= 3:
                # Join all middle parts until '_crop.tif'
                description = "_".join(parts[2:]).replace("_crop.tif", "")
                
                record = {
                    "city": city_name,
                    "tile_id": tile_id,
                    "crop_name": crop,
                    "crop_description": description,
                    "full_path": os.path.join(aoi_path, crop)
                }
                data.append(record)

crops_df = pd.DataFrame(data)

In [ ]:
# See how many crops for each city
city_crop_counts = crops_df.groupby("city").size().reset_index(name="count")
print(city_crop_counts)

In [ ]:
# See the mean number of crops per tile per city
city_tile_crop_counts = crops_df.groupby(["city", "tile_id"]).size().reset_index(name="count")
city_tile_crop_counts = city_tile_crop_counts.groupby("city")["count"].mean().reset_index(name="mean")
print(city_tile_crop_counts)


In [ ]:
# See how many tiles per city have crops
city_tile_crop_counts = crops_df.groupby(["city", "tile_id"]).size().reset_index(name="count")
city_tile_crop_counts = city_tile_crop_counts.groupby("city").size().reset_index(name="count")
print(city_tile_crop_counts)

In [ ]:
# See how many tiles per city have at least 2 crops
city_tile_crop_counts = crops_df.groupby(["city", "tile_id"]).size().reset_index(name="count")
city_tile_crop_counts = city_tile_crop_counts[city_tile_crop_counts["count"] >= 2].groupby("city").size().reset_index(name="count")
print(city_tile_crop_counts)
print(f"Total number of tiles with at least 2 crops: {city_tile_crop_counts['count'].sum()}")

In [ ]:
# Check how many tiles we have in total
tile_counts = crops_df.groupby(["city", "tile_id"]).size().reset_index(name="count")
print(tile_counts)